# AksaraLine 2/2 — train the recognizers

GPU. Consumes the corpus kernel's output, so no rendering happens here.

`06_run_matrix.py` skips cells that already have a `result.json`, so if the
session is cut short, re-running with the previous output attached resumes
rather than starting over.


In [ ]:
import os, sys, subprocess, zipfile, time
from pathlib import Path

BRANCH  = 'aksara-seq'
REPO    = Path('/kaggle/working/aksara_OCR')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/phoenixfin/aksantara-ocr.git',
                    str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'aksara_seq' / 'src'))
print('commit:', subprocess.run(['git','rev-parse','--short','HEAD'],
      capture_output=True, text=True).stdout.strip())


In [ ]:
INPUT = Path('/kaggle/input')

# Same lesson as the corpus kernel: do not assume the mount path. Find the
# corpus by looking for the directory layout it actually has.
def find_corpus(root, depth=5):
    frontier, seen = [root], []
    for _ in range(depth):
        nxt = []
        for d in frontier:
            try:
                nxt += [q for q in d.iterdir() if q.is_dir()]
            except OSError:
                pass
        seen += nxt
        frontier = nxt
    for d in seen:
        if d.name == 'v1' and (d / 'dataset_meta.json').is_file():
            return d
    return None

CORPUS = find_corpus(INPUT)
assert CORPUS is not None, (
    'corpus not found under /kaggle/input; attach the aksaraline-corpus '
    'kernel output. Present: '
    + str(sorted(q.name for q in INPUT.iterdir()) if INPUT.is_dir() else []))
RECOG = Path('/kaggle/working/build/recog')
print('corpus:', CORPUS)

for s in ['Sunda', 'Jawa', 'Bali', 'Lontara']:
    n = sum(1 for _ in (CORPUS / s / 'train' / 'images').glob('*.png'))
    print(f'  {s:9s} {n} train lines')
    assert n > 0, f'{s} has no rendered lines'

import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## The matrix

Four scripts × two label spaces. **Epochs are deliberately low for the first
run**: the point is to confirm the GPU path end to end and measure real
per-cell wall-clock, so the budget for a full run can be set from data rather
than from my estimate. Raise `--epochs` once the timings are known.


In [ ]:
!python aksara_seq/scripts/06_run_matrix.py     --corpus {CORPUS} --out {RECOG}     --epochs 12 --batch-size 32 --num-workers 2


In [ ]:
import csv
rows = list(csv.DictReader(open(RECOG / 'matrix.csv', encoding='utf-8')))
cols = ['script','head','test_ser','test_wer','test_line_acc',
        'onset_error','vowel_error','ser_clean','ser_heavy','minutes']
print(' '.join(f'{c:>13s}' for c in cols))
for r in rows:
    print(' '.join(f'{r.get(c,""):>13s}' for c in cols))
